# Kappa Foundation Analysis — Code Explanations and Assignment Check

This companion notebook explains every code cell in [`analysis.ipynb`](analysis.ipynb), and the extra audit cell in [`analysis_audited.ipynb`](analysis_audited.ipynb). It describes the purpose of each block, why the method is used, which professor-assigned evidence it supports, and any important caveats.

This is a review and learning guide, not a replacement for the executable analysis notebook. It contains no executable analysis code. The requirement mapping is based on the assignment brief in `docs/assignment/BFIN-WS2026-Group_Assignment.pdf` (especially its evidence checklist and reproducibility/submission sections).


## Overall assessment

The analysis notebook implements most of the quantitative evidence requested: a full allocation, bond and commitment cash-flow analysis, portfolio construction and comparison, factor models, a frozen out-of-sample evaluation, and traceable outputs. The data-processing path checks dates, units, completeness, and provenance. Two material limitations remain:

1. **Candidate eligibility is not fully evidenced.** The saved stock citation is a secondary S&P 500 snapshot dated 28 December 2022, not the 31 December case date. Dated case-date listing and mandate evidence for SPY and USMV is absent. The notebook discloses this; it does not satisfy the case-date eligibility requirement completely.
2. **The analysis is not the complete course submission.** The brief additionally requires a committee memo, slide deck plus editable source, the initial decision record and a decision log, contribution/check statements and critiques, and an AI-use disclosure. Several of these are not present in this repository. The decision record is explicitly a draft.

Other key caveats: inflation, ERP, risk aversion, the interpretation of a 15% loss concern, the per-security cap, cash carry, and trading costs are team assumptions, not values prescribed by the professor. The 2023–2025 evaluation is a retrospective check, not proof the 2030 real-value goal will be met.


## Requirement key

- **R1 — Mandate:** USD 12m allocation, both fixed payments, real-value goal, interpreted loss concern, quantitative risk/return preference and risk-aversion sensitivity.
- **R2 — Bonds/funding:** bond valuation and yields, incomplete spot curve, duration/convexity versus exact repricing, parallel and curve-shape scenarios, and relationship to commitments.
- **R3 — Portfolio:** recommended allocation and credible rejected alternative; compare sample-covariance and single-index risk on the same universe; explain forecasts, constraints, implementation, and sensitivity.
- **R4 — Market/factors:** market-only and at least one Lecture 7 multifactor model for the growth portfolio and two or more securities; interpret exposures, uncertainty, and alpha.
- **R5 — Evidence against:** frozen strategy versus benchmark in 2023–2025; quantify an assumption threshold that reverses an important conclusion; include adverse as well as favourable results.
- **R6 — Data/reproducibility:** use monthly data and the required windows, preserve raw downloads and provenance, document transformations, run without internet from saved raw data, include retrieval code, and trace outputs.
- **R7 — Complete submission:** memo, presentation, reproducible project/data, dated decision record and decision log, contribution/check statements, critiques, and AI disclosure.


## Code cell 1 — project setup and policy assumptions

**What it does.** Imports JSON/path, numerical, tabular, plotting, statistical, optimization, and display tools. It searches the working directory and its parents for the project root, loads the ticker universe from `config.json`, sets the raw-data and output paths, and defines the policy parameters: fund size, commitments, real goal, inflation, ERP, risk aversion, loss interpretation, liquidity reserve, security cap, transaction costs, estimation/evaluation windows, benchmark, and factor-test securities. `save_table` and `save_fig` centralize creation of named CSV/PNG evidence under the selected output tag.

**Why.** Centralizing assumptions reduces the chance that different analysis sections silently use different values. Locating the root from the notebook path allows execution from the repository or notebooks folder. Named output helpers make results auditable and reproducible.

**Requirement match.** Strongly supports R1 and R6; it establishes inputs later used for R2–R5. The chosen 2.5% inflation, 5% ERP, risk aversion 4, 15% one-year 95% total-fund-loss interpretation, 25% security cap, and 10 bp costs are **team choices that require justification/approval**, not professor-prescribed values. The loss measure is an interpretation of an ambiguous concern, not a trustee-approved hard limit.


## Code cell 3 — raw market/factor data, monthly returns, and sample checks

**What it does.** Reads each saved Yahoo CSV, normalizes its dates, and rejects duplicate/unsorted days or missing adjusted/ordinary closing prices. It creates month-end total returns from `Adj Close` and price-only returns from `Close`, retaining the two series visibly rather than forcing them to agree. It reads the CSV inside the saved Kenneth French ZIP, checks that it has one member and monthly date rows, converts the factors from percentages to decimals, maps source missing-value sentinels to missing values, and validates ordering, units, and completeness. It then enforces the frozen 2015-01–2022-12 estimation window (96 months) and separate 2023-01–2025-12 evaluation window, aligns returns and factors, rejects missing/non-finite estimation data, subtracts RF for excess returns, and writes `T01_data_quality.csv`.

**Why.** The assignment explicitly requires monthly data, a common estimation sample of at least 60 months ending December 2022, a separate evaluation window, visible return construction, and no invented fill-ins. Fail-fast checks prevent quietly using misaligned observations or percentage values as decimal returns.

**Requirement match.** Strong R6 coverage, including return construction and the estimation/evaluation windows; it provides input data for R3–R5. This does **not** establish case-date security eligibility. The input files are downloaded beforehand; this cell itself is offline.


## Code cell 4 — source-record hash verification

**What it does.** Loads `data/raw/source_record.json`, calculates SHA-256 for every manifest-listed file, compares each result with the recorded digest, stops on a mismatch, and writes `T00_source_record.csv` with source identifiers and hash status.

**Why.** A provenance manifest is only useful if the files actually match it. A failed hash signals a changed, corrupted, or mismatched raw input and should be investigated before trusting analysis outputs.

**Requirement match.** Directly supports R6: source integrity, traceability, and reproducibility. The block contains a second, identical hash-failure `if` in the original; it was redundant, so one copy has been removed from both executable notebooks. This has no effect on results.


## Code cell 6 — candidate universe, roles, and eligibility disclosure

**What it does.** Assigns each of ten stocks and two ETFs a sector/role, adds a human-readable eligibility claim and source reference, and saves `T02_universe.csv`. For stocks it explicitly records that the archived constituent list is dated 28 December 2022 and does not independently prove membership through the 31 December case date. For SPY and USMV it explicitly records that case-date listing/mandate documentation was not saved.

**Why.** The universe should have defensible economic roles and sector breadth. Explicitly documenting evidence limits is preferable to inferring historic eligibility from modern ticker symbols or price history.

**Requirement match.** Partially supports R3's candidate-universe construction (10 stocks across multiple sectors plus two different ETF mandates). It **does not fully meet R3's case-date eligibility-evidence requirement**. Find authoritative dated primary sources or revise the universe and evidence before representing eligibility as verified. Trading history proves price coverage, not index membership, listing, or ETF mandate.


## Code cell 8 — bond valuation, spot curve, duration, and convexity

**What it does.** Enters the five fictional bond terms and case prices from the assignment. Constructs annual coupon/principal cash flows, solves each yield to maturity, and bootstraps spot rates from bond prices in maturity order. It linearly interpolates between observed maturity knots to obtain a curve for years 1–8, reports implied forwards and identifies which maturities are actual quote knots in `T03_spot_curve.csv`. It then calculates Macaulay and modified duration, convexity, and numerical finite-difference checks in `T04_bond_analytics.csv`. `T05_duration_vs_exact.csv` compares duration-only and duration-plus-convexity estimates against exact repricing for several yield/curve shocks.

**Why.** The case provides executable fictional bond prices and contractual cash flows, but not a complete spot curve or resale history. Bootstrapping uses the provided quotes; interpolation fills unquoted maturities transparently. Numerical repricing tests whether analytical sensitivities behave as expected.

**Requirement match.** Strong R2 coverage for valuation, yields, incomplete curve treatment, duration/convexity, and exact repricing. Interpolation and shock sizes are modeling choices that should be justified. This block does not fabricate a historical bond-price series, as required.


## Code cell 9 — commitment-matching bond portfolio and rate/shape scenarios

**What it does.** Creates annual cash flows through year five, maps the 2024 and 2027 obligations into the cash-flow schedule, and solves a non-negative linear program minimizing initial bond purchase cost subject to cumulative bond cash inflows covering cumulative payments. It rounds unit counts up, checks that no interim cash balance is negative under zero cash carry, and writes `T06_funding_cashflows.csv` and `T07_bond_holdings.csv`. It then reprices the selected assets and payment liabilities under a base curve, parallel +/-100 bp and +250 bp shocks, and steepener/flattener curve-shape shocks, saving `T08_rate_scenarios_funding.csv`. It reports PV and duration context for assets and liabilities.

**Why.** The mandatory payments are fixed nominal obligations. Matching them with contractual cash flows is more relevant than using historical bond returns. A linear program makes the cost-minimizing funding choice explicit, while scenarios show how an assumed curve change affects asset/liability value.

**Requirement match.** Strong R2 coverage for payment funding, relationship to obligations, and parallel/shape scenarios. The model assumes 0% cash carry, default-free case bonds, and a limited five-year usable horizon; Epsilon is excluded because its later cash flows would require a resale-price assumption. The small residual cash balances are not a substantial liquidity cushion. Scenario magnitudes and reinvestment assumptions remain team choices.


## Code cell 11 — growth sleeve and real-value hurdle

**What it does.** Uses the one-year spot rate as the selected risk-free rate. Calculates the investable growth sleeve after bond cost and the cash reserve, then estimates one-way initial trading costs and residual cash. For alternative inflation assumptions (2%, 2.5%, 3%, and 4%) it computes the 2030 nominal equivalent of the USD 8m real goal and the annual growth return required to reach it, saving `T09_goal_requirements.csv`.

**Why.** This translates the trustees' purchasing-power goal into a nominal target and a hurdle that can be compared with portfolio forecasts. Separating the bond sleeve, growth sleeve, cash and transaction cost also supports a full-fund reconciliation.

**Requirement match.** Supports R1 and R3. The inflation assumption and zero cash return are not supplied by the professor; alternatives are shown, but the trustees/team must defend the base case.


## Code cell 13 — expected returns, risk models, and portfolio optimizer

**What it does.** Estimates each asset's beta and alpha from monthly excess returns against `Mkt-RF`, estimates residuals, constructs an annualized single-index covariance matrix and an annualized sample covariance matrix, and forms expected returns as the one-year spot rate plus beta times the assumed ERP. `mv_opt` maximizes mean-variance utility subject to a fully invested, long-only portfolio and a per-security cap. The cell checks optimizer success and resulting feasibility. It writes the risk-aversion sensitivity (`T10`), same-universe single-index versus sample-covariance weights and risk comparison (`T11`–`T12`), and an alternative highest-historical-Sharpe allocation over the full and last-60-month samples (`T13`). It plots the two risk-model allocations as `F01_weights.png`.

**Why.** The assignment requests a quantitative risk/return preference comparison, sensitivity to risk aversion, and a comparison of risk estimators on the same candidates. A documented optimizer translates forecasts and a utility function into explicit weights. The historical Sharpe alternative is a credible rejected approach and its time-window instability is made visible.

**Requirement match.** Strong R1 and R3 coverage. Forecasts, ERP, covariance estimation, no-short constraint, and 25% cap are assumptions/method choices that must be defended. The historical alternative uses data that are available in the retrospective analysis; clearly distinguish it from the frozen decision rule. A solver output is not itself a recommendation.


## Code cell 14 — risk diagnostics, sensitivity, and dollar allocation

**What it does.** Calculates worst rolling 12-month sleeve return, historical drawdown, the sleeve loss scaled to its share of total fund, a model-based one-year 95% loss estimate, expected return, and the real-goal hurdle (`T14`). It computes component risk contributions and betas (`T15`), varies ERP and the per-security cap (`T16`), constructs the bond/growth/cash/cost dollar allocation (`T17`), and asserts that all line items reconcile to USD 12m.

**Why.** This links the growth portfolio to whole-fund risk, shows what drives risk, tests key assumptions, and verifies the allocation arithmetic rather than leaving dollars unaccounted for.

**Requirement match.** Strong R1 and R3 coverage. Important qualification: the 15% loss concern is **reported as a diagnostic and printed comparison**, not enforced as a hard optimizer constraint. The modeled 95% loss uses a normal mean/volatility approximation; historical worst loss and drawdown are different measures. A result below the selected threshold does not prove safety or trustee approval. Sensitivity varies ERP/cap but does not establish that chosen values are objectively correct.


## Code cell 16 — CAPM/FF3/FF5 regressions and alpha checks

**What it does.** Defines CAPM, FF3 and FF5 specifications and checks the factor lists. `run_reg` enforces the exact 96-month estimation period, complete finite inputs, and required factors, then fits OLS with HAC standard errors (three lags). It applies the models to the growth portfolio and two selected stocks, subtracting RF once from total returns, and saves alphas, coefficients, t-statistics, fit statistics, observation counts and factor specifications in `T18_factor_regressions.csv`. It reports estimation-window factor premia (`T19`) and model-based expected-return comparisons (`T20`). Finally, it evaluates FF5 residual alpha in the separate 2023–2025 window with HAC inference (`T21`).

**Why.** This directly tests whether market-only and multifactor descriptions change the return explanation, exposes uncertainty around alpha, and keeps the initial estimation data separate from later evaluation data.

**Requirement match.** Strong R4 coverage: one market-only model and two multifactor models are reported for the growth portfolio and two stocks, exceeding the minimum model/security coverage. It supports R5 through the out-of-sample alpha check. Factor premia, the chosen French dataset vintage, and the relatively short 96-month sample warrant caution; estimated alpha is not proof of persistent skill. The professor did not prescribe CAPM/FF3/FF5 specifically.


## Code cell 18 — frozen strategy versus SPY out-of-sample evaluation

**What it does.** Selects exactly the 36 monthly observations in 2023–2025 and rejects missing/non-finite returns. `run_strategy` applies initial transaction costs, lets weights drift with asset returns, and rebalances only at December month-end if the maximum absolute weight drift exceeds 5 percentage points; it charges trading costs and tracks rebalance counts. It evaluates the recommended portfolio, SPY, the rejected historical-Sharpe portfolio and sample-covariance solution. `perf` reports cumulative/annualized return, volatility, Sharpe, drawdown and worst 12-month return. Results and monthly return paths go to `T22` and `T25`; a cumulative-value chart goes to `F02_evaluation.png`.

**Why.** A predeclared, separate period tests the frozen growth strategy against a simple investable benchmark without changing the initial decision based on future returns. Including adverse metrics and explicit transaction/rebalance assumptions makes the comparison more informative than reporting cumulative return alone.

**Requirement match.** Strong R5 coverage for the frozen strategy, benchmark and evaluation years; it also supports R6 output traceability. The evaluation is retrospective, only three years long, and cannot establish the eight-year mission goal. The 5 pp trigger and cost rate are policy assumptions and should match the dated decision record.


## Code cell 20 — reversal thresholds and goal distribution

**What it does.** Solves for the ERP and inflation values at which expected growth return crosses the required return (`T26_reversal_thresholds.csv`). It uses a lognormal approximation based on estimated growth mean/volatility to calculate goal probabilities over an ERP/inflation grid (`T23_goal_probability.csv`) and real-fund-value percentiles (`T24_real_value_distribution.csv`).

**Why.** The assignment asks for at least one plausible assumption change that could reverse an important decision, with a quantified threshold, and asks the team to make uncertainty around the longer-term goal visible.

**Requirement match.** Directly supports R5 and R1. The crossover is a model-based expected-return hurdle, not a guarantee that the goal will or will not be achieved. Goal probabilities depend strongly on the lognormal return approximation, selected growth-return model, inflation and assumed cash treatment; describe them as scenarios, not precise forecasts.


## Code cell 22 — policy freeze and final analysis assertions

**What it does.** Serializes the chosen assumptions, bond holdings, growth weights, dollar sleeve, benchmark, estimation/evaluation windows, and rebalance rule to `frozen_policy.json`. If that file already exists, it refuses to overwrite it with different values, requiring a new output tag for a revised mandate. It then checks data-download failures and hashes, regression model coverage/specifications/window, HAC reporting, rebalance counts, and weight sum, and prints the output inventory.

**Why.** The assignment requires a dated frozen initial policy and says revisions must be compared against it rather than retroactively replacing it. Final assertions catch important departures from that frozen specification.

**Requirement match.** Strong R5 and R6 coverage. The cell protects the JSON policy snapshot, but the notebook still writes other tables/figures to the current output directory before this final cell runs; use a new tag for revised work and preserve the original directory. This block does not substitute for the separately required decision log.


## Audited notebook extra code cell — final submission checks

**Where.** The additional last code cell in `analysis_audited.ipynb`; it is not present in the canonical `analysis.ipynb`.

**What it does.** Independently reopens the raw source manifest and frozen outputs, rechecks raw hashes and policy parameters, verifies required bond/goal tables and repricing tolerance, confirms CAPM/FF3/FF5 row coverage and estimation period, checks HAC out-of-sample statistics and the expected rebalance counts, and verifies the growth weights sum to one.

**Why.** This gives the audited copy a final, separate set of assertions rather than relying only on earlier calculations or printed messages.

**Requirement match.** Supports R6 and verifies pieces of R1–R5. It is validation, not extra economic analysis, and does not fix the universe-evidence or course-deliverable gaps. It expects the saved output tag and source data to exist, so run the main analysis first.


## Professor-requirement scorecard

| Requirement | Notebook coverage | Current assessment |
|---|---|---|
| USD 12m allocation, payment plan, real goal and explicit risk interpretation | Cells 1, 9, 11, 14 | Quantified; assumptions and risk-policy approval remain team responsibilities. |
| Bond prices/yields, incomplete curve, duration/convexity and exact repricing | Cells 8–9 | Covered with transparent interpolation and fictional contractual cash flows; scenario justification remains important. |
| 10 stocks across multiple sectors plus two ETFs and case-date eligibility | Cell 6 | Universe exists; case-date eligibility proof is incomplete, so this requirement is not fully satisfied. |
| Recommended portfolio, credible rejected alternative, covariance comparison, assumptions and sensitivity | Cells 13–14 | Covered quantitatively; inputs/constraints require written rationale. |
| Market-only and multifactor models for growth and at least two securities | Cell 16 | Covered with CAPM, FF3, FF5, 96 observations; explain short-sample uncertainty and avoid skill claims. |
| Frozen strategy vs benchmark, adverse metrics and quantified reversal threshold | Cells 18, 20, 22 | Covered in the analysis; 2023–2025 is not proof of 2030 goal success. |
| Raw source, provenance, retrieval, offline rebuild, and named outputs | Cells 3–4, 22; `src/` and `scripts/` | Covered for the included data snapshot; rerun validators after changing inputs. |
| Memo, slides, source, decision log, contributions/checks, critiques, AI disclosure | Not analysis code | Still requires separate team deliverables; see the README and audit report. |


## Suggested walkthrough

1. Read this guide first, then open `analysis.ipynb` beside it.
2. Match each listed cell number to its code cell in the original notebook; use the output names to locate generated evidence.
3. Open `docs/assignment/BFIN-WS2026-Group_Assignment.pdf` and compare the scorecard with the original assignment wording.
4. Before submission, obtain and archive dated eligibility evidence, confirm team approval of the modeling assumptions, and finish the non-notebook deliverables.
5. If any code or inputs change, execute the notebook from saved raw data and rerun `scripts/validate_data.py` and `scripts/validate_analysis.py --tag original`.
